# Mews NL → US: Treasury for Platforms — Production Test Plan

**Test Date:** _fill in before running_ (plan drafted 2026-08-19)  
**Platform:** Treasury EU: Belgium, `acct_1SPgP3EW60zLc0T7` (live-mode)  
**Corridor:** STEL (BE Platform) → SPC (US Connected Account)  
**Environment:** LIVE

## Scope

This notebook walks the **Mews EU → US Production Test Plan** end to end:
1. Onboarding & Compliance Plan validation (US CA under EU platform)
2. v2 Financial Account creation (USD, on the US CA)
3. Money movement — external (US rails: ACH, wire)
4. Money movement — internal (FA↔FA cross-entity, STEL↔SPC)
5. Payments — destination charges + OBO
6. Cross-Border Transfers (CBT)
7. Recipient / Payouts
8. Application Fees (AFF) — NL platform, US CAs

**⚠️ Out of scope:** Issuing is explicitly excluded from this test (separate validation track with the bank partner, DRI: John Piazza).

## Prerequisites (from the test plan)
- Platform (`acct_1SPgP3EW60zLc0T7`) gated for cross-region interop: multicurrency + cross-region FA distribution flag (Eng/TREX)
- Platform has v2 FA enabled
- US bank partner (SPC) coverage confirmed available for CAs under this platform
- Test US CA created (or reuse existing) — default below: `acct_1Tbg1nINWqC43lYI`
- Platform FA (EUR) funded with a small amount (e.g. €100)
- CA FA (USD) funded with a small amount (e.g. $100)

**Notes:**
- This is LIVE mode — test helpers will NOT work. Funding steps require real bank transfers or real card charges.
- Several checks (Processing Entity, Financial Sponsor, Compliance Plan, onboarding ownership, CBT fee bps, billing entity) live in internal tooling (go/views, User Billing ledger) rather than the public API — those cells print reminders instead of API calls where that's the case. Record the manual result in the ✅/❌ column of the source doc.
- Settlement timing: expect **T+1** (not instant) for cross-entity FA↔FA transfers — confirm with Megan Li if timing looks off.
- Compare live results against the sandbox penny test (20/20 passing, 2026-07-08) — any divergence is a signal worth escalating to #spend-and-earn (SPEAR).

In [ ]:
import requests
import json
from typing import Dict, Any

# ============================================================================
# CONFIGURATION - UPDATE THESE VALUES
# ============================================================================

SECRET_KEY = "sk_live_xxxx"  # Live secret key for the Treasury EU platform (acct_1SPgP3EW60zLc0T7)
API_VERSION = "2026-08-26.preview"

# IDs table from the test plan (fill in as you create each object)
test_data = {
    "platform_account_id": "acct_1SPgP3EW60zLc0T7",   # Mews NL, live-mode
    "platform_fa_id": "fa_65TwcHvlu49mspR6PrC16TZ4MPQ79Clh3eLzGNN0GbQ6O8",  # Platform multicurrency FA (EUR/USD/GBP)
    "platform_fa_eur_id": "fa_65TwcHvlu49mspR6PrC16TZ4MPQ79Clh3eLzGNN0GbQ6O8",                        # Same FA as platform_fa_id when multicurrency
    "platform_fa_usd_id": "fa_65TwcHvlu49mspR6PrC16TZ4MPQ79Clh3eLzGNN0GbQ6O8",                         # Same FA as platform_fa_id when multicurrency
    "platform_fa_gbp_id": "fa_65TwcHvlu49mspR6PrC16TZ4MPQ79Clh3eLzGNN0GbQ6O8",                         # Same FA as platform_fa_id when multicurrency
    "us_ca_id": "acct_1UFdgtIjc7BMQplW",               # US hotel test CA (Section 1.1) 
    "us_ca_id1": "acct_1Tbg1nINWqC43lYI",                # Exising US account
    "us_ca_fa_usd_id": "fa_65VPcT1j296iiAuzC1d16VP1eFQgSQW4NIGeZiDGVy4Emm", # CA's USD FA (Section 2.1)
    "us_ca_external_bank_id": "usba_61VQ4PsbO0LX7TN8u16VP1eFQgSQW4NIGeZiDGVy4Qb2",
    "us_ca_attached_external_bank_id": "ba_1UGaEfIjc7BMQplWfckiCBCC",  # CA's external US bank account (Section 7.1)
    "test_payment_method_id": "pm_1UGyMWEW60zLc0T7kUPJ2UKx",  # Card token for test charges (Section 5)
    "test_customer_id": "acct_1UGyIcEaCOBZn9dt",         # Customer object for charges (Section 5)
    "destination_charge_id": None,                       # Section 5.1
    "cbt_transfer_linked_id": None,                      # Section 6.3
    "cbt_transfer_unlinked_id": None,                    # Section 6.4
    "sct_charge_id": None,                                # Section 8.4
    "sct_transfer_id": None,                              # Section 8.4
}

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def print_section(title: str):
    print("\n" + "="*80)
    print(f"  {title}")
    print("="*80 + "\n")

def manual_check(instructions: str):
    """For scenarios verified via go/views or internal ledgers, not the public API."""
    print("🔎 MANUAL CHECK REQUIRED (not exposed via public API):")
    print(instructions)
    print("\nRecord the result in the ✅/❌ column of the test plan doc.")

def print_result(response: requests.Response, show_full: bool = False):
    print(f"Status: {response.status_code}")
    if response.status_code >= 400:
        print("❌ ERROR")
        try:
            print(json.dumps(response.json(), indent=2))
        except Exception:
            print(response.text)
        return None
    print("✅ SUCCESS")
    data = response.json()
    if show_full:
        print(json.dumps(data, indent=2))
    else:
        for k in ("id", "object", "status", "balance"):
            if k in data:
                print(f"{k.capitalize()}: {data[k]}")
    return data

def v2_headers(stripe_account: str = None, stripe_context: str = None) -> Dict[str, str]:
    headers = {
        "Authorization": f"Bearer {SECRET_KEY}",
        "Stripe-Version": API_VERSION,
        "Content-Type": "application/json"
    }
    if stripe_account:
        headers["Stripe-Account"] = stripe_account
    if stripe_context:
        headers["Stripe-Context"] = stripe_context
    return headers

def v1_auth() -> tuple:
    return (SECRET_KEY, "")

def print_fa_bank_details(addr: dict):
    """Print routing/account number or IBAN so you can send a live bank transfer."""
    creds = addr.get("credentials") or {}
    cred_type = creds.get("type")
    print("\n🏦 Bank details for inbound transfer")
    print("=" * 60)
    print(f"  Financial Address ID: {addr.get('id')}")
    print(f"  Status:               {addr.get('status')}")
    print(f"  Currency:             {(addr.get('currency') or '').upper()}")
    print(f"  Type:                 {cred_type}")

    if cred_type == "us_bank_account":
        us = creds.get("us_bank_account") or {}
        acct = us.get("account_number") or (f"********{us.get('last4')}" if us.get("last4") else None)
        print(f"  Bank name:            {us.get('bank_name')}")
        print(f"  Routing number:       {us.get('routing_number')}")
        print(f"  Account number:       {acct}")
        if us.get("bic"):
            print(f"  BIC / SWIFT:          {us.get('bic')}")
        if us.get("iban"):
            print(f"  IBAN:                 {us.get('iban')}")
        if us.get("account_holder_name"):
            print(f"  Account holder:       {us.get('account_holder_name')}")
    elif cred_type == "gb_bank_account":
        gb = creds.get("gb_bank_account") or {}
        acct = gb.get("account_number") or (f"********{gb.get('last4')}" if gb.get("last4") else None)
        print(f"  Sort code:            {gb.get('sort_code')}")
        print(f"  Account number:       {acct}")
        if gb.get("account_holder_name"):
            print(f"  Account holder:       {gb.get('account_holder_name')}")
    elif cred_type in ("iban", "sepa_bank_account"):
        details = creds.get(cred_type) or creds.get("iban") or creds.get("sepa_bank_account") or {}
        print(f"  IBAN:                 {details.get('iban')}")
        if details.get("bic"):
            print(f"  BIC:                  {details.get('bic')}")
        if details.get("account_holder_name"):
            print(f"  Account holder:       {details.get('account_holder_name')}")
        if details.get("last4") and not details.get("iban"):
            print(f"  IBAN last4:           ********{details.get('last4')}")
    else:
        print(json.dumps(creds, indent=2))
    print("=" * 60)

def list_financial_accounts(stripe_account: str):
    """Return FA list, or None if listing failed (do not treat that as empty)."""
    accounts = []
    url = "https://api.stripe.com/v2/money_management/financial_accounts"
    params = {"limit": 20}
    while url:
        response = requests.get(
            url,
            headers=v2_headers(stripe_account=stripe_account),
            params=params,
        )
        params = None
        if response.status_code >= 400:
            print_result(response, show_full=True)
            return None
        body = response.json() or {}
        accounts.extend(body.get("data") or [])
        url = body.get("next_page_url")
        if url and url.startswith("/"):
            url = "https://api.stripe.com" + url
    return accounts

def list_financial_addresses(stripe_account: str, financial_account_id: str = None):
    includes = [
        "credentials.us_bank_account.account_number",
        "credentials.gb_bank_account.account_number",
        "credentials.sepa_bank_account.account_number",
    ]
    params = {f"include[{i}]": value for i, value in enumerate(includes)}
    if financial_account_id:
        params["financial_account"] = financial_account_id
    response = requests.get(
        "https://api.stripe.com/v2/money_management/financial_addresses",
        headers=v2_headers(stripe_account=stripe_account),
        params=params,
    )
    if response.status_code >= 400:
        params.pop("include[2]", None)
        response = requests.get(
            "https://api.stripe.com/v2/money_management/financial_addresses",
            headers=v2_headers(stripe_account=stripe_account),
            params=params,
        )
    if response.status_code >= 400:
        print_result(response, show_full=True)
        return []
    return (response.json() or {}).get("data") or []

PLATFORM_CURRENCIES = ["eur", "usd", "gbp"]
PLATFORM_ADDRESS_TYPES = {
    "eur": "sepa_bank_account",
    "usd": "us_bank_account",
    "gbp": "gb_bank_account",
}
ADDRESS_TYPE_ALIASES = {
    "sepa_bank_account": {"sepa_bank_account", "iban"},
    "us_bank_account": {"us_bank_account"},
    "gb_bank_account": {"gb_bank_account"},
}

def fa_currencies(fa: dict):
    return [(c or "").lower() for c in ((fa.get("storage") or {}).get("holds_currencies") or [])]

def pick_platform_multicurrency_fa(fas):
    open_fas = [fa for fa in fas if fa.get("status") != "closed"]
    if not open_fas:
        return None
    return max(open_fas, key=lambda fa: len(set(fa_currencies(fa)) & set(PLATFORM_CURRENCIES)))

def add_currencies_to_fa(stripe_account: str, fa: dict, currencies):
    current = fa_currencies(fa)
    needed = [c for c in currencies if c not in current]
    if not needed:
        print(f"ℹ️ {fa['id']} already holds {current}")
        return fa
    merged = list(dict.fromkeys(current + list(currencies)))
    print(f"Adding currencies {needed} to {fa['id']} → {merged}")
    response = requests.post(
        f"https://api.stripe.com/v2/money_management/financial_accounts/{fa['id']}",
        headers=v2_headers(stripe_account=stripe_account),
        json={"storage": {"holds_currencies": merged}},
    )
    updated = print_result(response, show_full=True)
    return updated or fa

def ensure_currency_addresses(stripe_account: str, fa: dict, currency_types: dict):
    addrs = list_financial_addresses(stripe_account, fa["id"])
    existing_types = set()
    existing_currencies = set()
    for addr in addrs:
        creds = addr.get("credentials") or {}
        existing_types.add(creds.get("type") or addr.get("type"))
        existing_currencies.add((addr.get("currency") or "").lower())
    for currency, addr_type in currency_types.items():
        aliases = ADDRESS_TYPE_ALIASES.get(addr_type, {addr_type})
        if aliases & existing_types or currency in existing_currencies:
            print(f"ℹ️ {currency.upper()} address already present ({addr_type})")
            continue
        print(f"Creating {addr_type} address for {currency.upper()} on {fa['id']}")
        create_response = requests.post(
            "https://api.stripe.com/v2/money_management/financial_addresses",
            headers=v2_headers(stripe_account=stripe_account),
            json={"type": addr_type, "financial_account": fa["id"]},
        )
        print_result(create_response, show_full=False)
    addrs = list_financial_addresses(stripe_account, fa["id"])
    if not addrs:
        print("⚠️ No financial address to print.")
        return []
    for addr in addrs:
        print_fa_bank_details(addr)
    return addrs

def address_type_for_fa(fa: dict) -> str:
    country = (fa.get("country") or "").upper()
    currs = [(c or "").lower() for c in ((fa.get("storage") or {}).get("holds_currencies") or [])]
    if country == "US" or "usd" in currs:
        return "us_bank_account"
    if country in ("GB", "UK") or "gbp" in currs:
        return "gb_bank_account"
    return "sepa_bank_account"

def print_financial_account(fa: dict):
    currs = ((fa.get("storage") or {}).get("holds_currencies")) or []
    print(f"\n💰 FA {fa.get('id')}")
    print(f"  display_name: {fa.get('display_name')}")
    print(f"  status:       {fa.get('status')}")
    print(f"  country:      {fa.get('country')}")
    print(f"  currencies:   {currs}")
    for btype, currencies in (fa.get("balance") or {}).items():
        if not isinstance(currencies, dict):
            continue
        for ccy, amt in currencies.items():
            if isinstance(amt, dict) and "value" in amt:
                print(f"  {ccy.upper()} {btype}: {amt['value'] / 100:.2f}")

def ensure_financial_address(stripe_account: str, fa: dict):
    addrs = list_financial_addresses(stripe_account, fa["id"])
    if not addrs:
        addr_type = address_type_for_fa(fa)
        print(f"No financial address on {fa['id']} — creating type={addr_type}")
        create_response = requests.post(
            "https://api.stripe.com/v2/money_management/financial_addresses",
            headers=v2_headers(stripe_account=stripe_account),
            json={"type": addr_type, "financial_account": fa["id"]},
        )
        created = print_result(create_response, show_full=False)
        if created:
            addrs = list_financial_addresses(stripe_account, fa["id"]) or [created]
    if not addrs:
        print("⚠️ No financial address to print.")
        return []
    for addr in addrs:
        print_fa_bank_details(addr)
    return addrs

print("✅ Setup complete!")
print(f"API Version: {API_VERSION}")
print(f"Secret Key: {SECRET_KEY[:15]}...")
print(f"Platform: {test_data['platform_account_id']}")

✅ Setup complete!
API Version: 2026-08-26.preview
Secret Key: sk_live_51SPgP3...
Platform: acct_1SPgP3EW60zLc0T7


---
# Section 1: Onboarding & Compliance Plan Validation

Proves that a US-domiciled CA created under the NL platform gets US compliance treatment (SPC processing entity, US bank partner, full ToS, US-prefixed compliance plan, active capabilities, platform-owned onboarding).

In [ ]:
print_section("1.1 Create US CA under EU platform")

# Skip creation if you're reusing the existing test_data['us_ca_id'].
payload = {
    "contact_email": "us-hotel-test@mews.com",
    "display_name": "Mews US Hotel Test CA",
    "identity": {
        "country": "us",
        "entity_type": "individual"
    },
    "dashboard": "none",
    "defaults": {
        "responsibilities": {       
        "fees_collector": "application",
        "losses_collector": "application",
        }
    },
    "include": [
        "configuration.money_manager",
        "configuration.recipient",
        "requirements",
        "identity"
    ],
    "configuration": {
        "merchant": {
             "capabilities": {
                "card_payments": {"requested": True}
             }
        },
        "money_manager": {
            "capabilities": {
                "business_storage": {
                    "inbound": {"usd": {"requested": True}},
                    "outbound": {"usd": {"requested": True}}
                },
                "received_credits": {
                    "bank_accounts": {"requested": True}
                },
                "inbound_transfers": {
                    "bank_accounts": {"requested": True}
                },
                "outbound_payments": {
                    "bank_accounts": {"requested": True},
                    "cards": {"requested": True},
                    "financial_accounts": {"requested": True}
                },
                "outbound_transfers": {
                    "bank_accounts": {"requested": True},
                    "financial_accounts": {"requested": True}
                }
            }
        },
        "recipient": {
            "capabilities": {
                "stripe_balance": {
                    "stripe_transfers": {"requested": True}
                }
            }
        }
    }
}

response = requests.post(
    "https://api.stripe.com/v2/core/accounts",
    headers=v2_headers(stripe_account=test_data['platform_account_id']),
    json=payload
)

account = print_result(response, show_full=False)

if account:
    test_data['us_ca_id'] = account['id']
    print(f"\n📝 US CA ID: {test_data['us_ca_id']}")
    print(f"Country: {account.get('identity', {}).get('country', 'N/A')}")

    # POST /v2/core/account_links — Stripe-Version: 2026-08-26.preview
    # https://docs.stripe.com/api/v2/core/account-links/create?api-version=2026-08-26.preview
    print_section("1.1b Create v2 Account Link for onboarding")
    configs = account.get("applied_configurations") or ["merchant", "money_manager", "recipient"]
    link_response = requests.post(
        "https://api.stripe.com/v2/core/account_links",
        headers=v2_headers(),
        json={
            "account": test_data["us_ca_id"],
            "use_case": {
                "type": "account_onboarding",
                "account_onboarding": {
                    "configurations": configs,
                    "refresh_url": "https://example.com/refresh",
                    "return_url": "https://example.com/return",
                },
            },
        },
    )
    link = print_result(link_response, show_full=True)
    if link:
        print(f"\n🔗 Onboarding URL: {link.get('url')}")
        print(f"Expires at: {link.get('expires_at')}")
        print("Open this URL to complete hosted onboarding.")

In [ ]:
print_section("1.1c Update US CA capabilities (money_manager + recipient)")

# POST /v2/core/accounts/{id} — request extra rails on the existing US CA
# (outbound_payments.cards / financial_accounts, outbound_transfers.financial_accounts, recipient)
if not test_data["us_ca_id"]:
    print("⚠️ No US CA ID. Run 1.1 first or set test_data['us_ca_id'].")
else:
    payload = {
        "include": [
            "configuration.money_manager",
            "configuration.recipient",
            "requirements",
        ],
        "configuration": {
            "money_manager": {
                "capabilities": {
                    "business_storage": {
                        "inbound": {"usd": {"requested": True}},
                        "outbound": {"usd": {"requested": True}},
                    },
                    "received_credits": {
                        "bank_accounts": {"requested": True},
                    },
                    "inbound_transfers": {
                        "bank_accounts": {"requested": True},
                    },
                    "outbound_payments": {
                        "bank_accounts": {"requested": True},
                        "cards": {"requested": True},
                        "financial_accounts": {"requested": True},
                    },
                    "outbound_transfers": {
                        "bank_accounts": {"requested": True},
                        "financial_accounts": {"requested": True},
                    },
                }
            },
            "recipient": {
                "capabilities": {
                    "stripe_balance": {
                        "stripe_transfers": {"requested": True},
                    },
                },
            },
        },
    }

    response = requests.post(
        f"https://api.stripe.com/v2/core/accounts/{test_data['us_ca_id']}",
        headers=v2_headers(stripe_account=test_data["platform_account_id"]),
        json=payload,
    )
    account = print_result(response, show_full=False)
    if account:
        config = account.get("configuration") or {}
        mm = (config.get("money_manager") or {}).get("capabilities") or {}
        rec = (config.get("recipient") or {}).get("capabilities") or {}
        print(f"\n📝 Updated US CA: {account.get('id')}")
        print(f"applied_configurations: {account.get('applied_configurations')}")
        print("\n📋 money_manager capabilities:")
        for cap, details in mm.items():
            print(f"  {cap}: {json.dumps(details)}")
        print("\n📋 recipient capabilities:")
        print(json.dumps(rec, indent=2) if rec else "  (not applied)")


In [ ]:
print_section("1.2 - 1.7 Compliance & capabilities checks")

# GET /v2/core/accounts/{id} — Stripe-Version: 2026-08-26.preview
# https://docs.stripe.com/api/v2/core/accounts/retrieve?api-version=2026-08-26.preview
if not test_data['us_ca_id']:
    print("⚠️ No US CA ID. Run 1.1 first.")
else:
    response = requests.get(
        f"https://api.stripe.com/v2/core/accounts/{test_data['us_ca_id']}",
        headers=v2_headers(),
        params={
            "include[0]": "defaults",
            "include[1]": "identity",
            "include[2]": "configuration.money_manager",
            "include[3]": "configuration.recipient",
            "include[4]": "requirements",
        },
    )
    account = print_result(response, show_full=True)
    if account:
        identity = account.get("identity") or {}
        defaults = account.get("defaults") or {}
        money_manager = (account.get("configuration") or {}).get("money_manager") or {}
        recipient = (account.get("configuration") or {}).get("recipient") or {}
        caps = money_manager.get("capabilities") or {}

        print("\n📋 Account summary:")
        print(f"  display_name: {account.get('display_name')}")
        print(f"  identity.country: {identity.get('country')}")
        print(f"  identity.entity_type: {identity.get('entity_type')}")
        print(f"  applied_configurations: {account.get('applied_configurations')}")
        print(f"  defaults.responsibilities: {defaults.get('responsibilities')}")

        print("\n📋 money_manager capabilities:")
        if not caps:
            print("  (none returned — check include / applied configurations)")
        for cap, details in caps.items():
            print(f"  {cap}: {json.dumps(details)}")

        rec_caps = recipient.get("capabilities") or {}
        print("\n📋 recipient capabilities:")
        print(json.dumps(rec_caps, indent=2) if rec_caps else "  (not applied)")

        reqs = account.get("requirements")
        if reqs:
            print("\n📋 requirements:")
            print(json.dumps(reqs, indent=2))

    manual_check(
        "1.2 Processing Entity = SPC        -> go/views should show 'US platform-controlled', Processing Entity = Stripe Payments Company\n"
        "1.3 Financial Sponsor = US bank    -> go/views should show PNC or equivalent US bank partner\n"
        "1.4 ToS agreement type = full      -> Account summary should show service_agreement: full\n"
        "1.5 US Compliance Plan assigned    -> Capabilities section should show US-prefixed CP (e.g. us_managed_*)\n"
        "1.7 Onboarding owned by Platform   -> go/views should show 'Onboarding owned by: Platform'"
    )

---
# Section 2: v2 Financial Account Creation

List existing Financial Accounts (and their financial addresses) first. Create only if none exist.

1. **Platform** — one multicurrency FA holding **EUR, USD, GBP**, with a financial address per currency (SEPA/IBAN, US ACH/wire, UK sort code)
2. **US CA** — USD FA under SPC + US bank address


In [2]:
print_section("2.0 List or create platform multicurrency FA + EUR/USD/GBP addresses")

if not test_data["platform_account_id"]:
    print("⚠️ No platform account ID.")
else:
    platform_id = test_data["platform_account_id"]
    fas = list_financial_accounts(platform_id)
    fa = None
    if fas is None:
        print("⚠️ Could not list platform FAs — not creating another.")
    elif fas:
        print(f"ℹ️ Found {len(fas)} platform FA(s). Listing, then using one multicurrency FA.")
        for existing in fas:
            print_financial_account(existing)
        fa = pick_platform_multicurrency_fa(fas)
        if not fa:
            print("⚠️ No open platform FA to reuse.")
        else:
            print(f"\n➡️ Using {fa['id']} as the platform multicurrency FA")
            fa = add_currencies_to_fa(platform_id, fa, PLATFORM_CURRENCIES)
            print_financial_account(fa)
            ensure_currency_addresses(platform_id, fa, PLATFORM_ADDRESS_TYPES)
    else:
        print("No platform FA — creating multicurrency storage FA (EUR/USD/GBP) + addresses.")
        response = requests.post(
            "https://api.stripe.com/v2/money_management/financial_accounts",
            headers=v2_headers(stripe_account=platform_id),
            json={
                "type": "storage",
                "storage": {"holds_currencies": PLATFORM_CURRENCIES},
                "display_name": "Mews NL Platform FA (EUR/USD/GBP)",
            },
        )
        fa = print_result(response, show_full=False)
        if fa:
            print_financial_account(fa)
            ensure_currency_addresses(platform_id, fa, PLATFORM_ADDRESS_TYPES)

    # if fa:
    #     test_data["platform_fa_id"] = fa["id"]
    #     test_data["platform_fa_eur_id"] = fa["id"]
    #     test_data["platform_fa_usd_id"] = fa["id"]
    #     test_data["platform_fa_gbp_id"] = fa["id"]

    print(f"\n📝 platform_fa_id:     {test_data.get('platform_fa_id')}")
    print(f"📝 platform_fa_eur_id: {test_data.get('platform_fa_eur_id')}")
    print(f"📝 platform_fa_usd_id: {test_data.get('platform_fa_usd_id')}")
    print(f"📝 platform_fa_gbp_id: {test_data.get('platform_fa_gbp_id')}")


  2.0 List or create platform multicurrency FA + EUR/USD/GBP addresses

ℹ️ Found 10 platform FA(s). Listing, then using one multicurrency FA.

💰 FA fa_65UvKLMW84uwJcDrs8416TZ4MPQ79Clh3eLzGNN0GbQHv6
  display_name: Platform FA for Sending
  status:       open
  country:      BE
  currencies:   ['cad', 'eur']
  CAD available: -0.38
  EUR available: -0.31
  CAD inbound_pending: 0.00
  EUR inbound_pending: 0.00
  CAD outbound_pending: 0.00
  EUR outbound_pending: 0.00

💰 FA fa_65UubEKGAo8HeLr7J3V16TZ4MPQ79Clh3eLzGNN0GbQQbA
  display_name: Financial account for expense management
  status:       open
  country:      BE
  currencies:   ['cad', 'eur', 'gbp', 'usd']
  CAD available: 1.33
  EUR available: 0.00
  GBP available: 0.50
  USD available: 0.00
  CAD inbound_pending: 0.00
  EUR inbound_pending: 0.00
  GBP inbound_pending: 0.00
  USD inbound_pending: 0.00
  CAD outbound_pending: 0.00
  EUR outbound_pending: 0.00
  GBP outbound_pending: 0.00
  USD outbound_pending: 0.00

💰 FA fa_65UryQz

In [3]:
print_section("Check platform FA available balance (multicurrency)")

fa_id = test_data.get("platform_fa_id") or test_data.get("platform_fa_eur_id")
if not fa_id:
    print("⚠️ No platform FA ID. Run 2.0 first.")
else:
    response = requests.get(
        f"https://api.stripe.com/v2/money_management/financial_accounts/{fa_id}",
        headers=v2_headers(),
    )
    fa = print_result(response, show_full=False)
    if fa:
        currs = ((fa.get("storage") or {}).get("holds_currencies")) or []
        available = ((fa.get("balance") or {}).get("available")) or {}
        print(f"\n💰 Platform FA: {fa.get('id')}")
        print(f"  display_name: {fa.get('display_name')}")
        print(f"  status:       {fa.get('status')}")
        print(f"  country:      {fa.get('country')}")
        print(f"  currencies:   {currs}")
        print("\n  Available balance:")
        if not available:
            print("    (none)")
        for ccy in ("eur", "usd", "gbp"):
            amt = available.get(ccy)
            if isinstance(amt, dict) and "value" in amt:
                print(f"    {ccy.upper()}: {amt['value'] / 100:.2f}")
            else:
                print(f"    {ccy.upper()}: (not held / no balance)")
        extras = [c for c in available if c not in ("eur", "usd", "gbp")]
        for ccy in extras:
            amt = available[ccy]
            if isinstance(amt, dict) and "value" in amt:
                print(f"    {ccy.upper()}: {amt['value'] / 100:.2f}")
        print("\n  Other buckets:")
        for btype, currencies in (fa.get("balance") or {}).items():
            if btype == "available" or not isinstance(currencies, dict):
                continue
            for ccy, amt in currencies.items():
                if isinstance(amt, dict) and "value" in amt:
                    print(f"    {ccy.upper()} {btype}: {amt['value'] / 100:.2f}")



  Check platform FA available balance (multicurrency)

Status: 200
✅ SUCCESS
Id: fa_65TwcHvlu49mspR6PrC16TZ4MPQ79Clh3eLzGNN0GbQ6O8
Object: v2.money_management.financial_account
Status: open
Balance: {'available': {'eur': {'value': 100, 'currency': 'eur'}, 'gbp': {'value': 799, 'currency': 'gbp'}, 'usd': {'value': 900, 'currency': 'usd'}}, 'inbound_pending': {'eur': {'value': 0, 'currency': 'eur'}, 'gbp': {'value': 0, 'currency': 'gbp'}, 'usd': {'value': 0, 'currency': 'usd'}}, 'outbound_pending': {'eur': {'value': 0, 'currency': 'eur'}, 'gbp': {'value': 0, 'currency': 'gbp'}, 'usd': {'value': 0, 'currency': 'usd'}}}

💰 Platform FA: fa_65TwcHvlu49mspR6PrC16TZ4MPQ79Clh3eLzGNN0GbQ6O8
  display_name: None
  status:       open
  country:      BE
  currencies:   ['eur', 'gbp', 'usd']

  Available balance:
    EUR: 1.00
    USD: 9.00
    GBP: 7.99

  Other buckets:
    EUR inbound_pending: 0.00
    GBP inbound_pending: 0.00
    USD inbound_pending: 0.00
    EUR outbound_pending: 0.00
    GBP

In [4]:
print_section("2.1 List or create US CA Financial Account (USD) + address")

if not test_data["us_ca_id"]:
    print("⚠️ No US CA ID. Run Section 1 first.")
else:
    fas = list_financial_accounts(test_data["us_ca_id"])
    if fas is None:
        print("⚠️ Could not list US CA FAs — not creating another.")
    elif fas:
        print(f"ℹ️ Found {len(fas)} FA(s) on US CA — not creating another.")
        usd_ids = []
        for fa in fas:
            print_financial_account(fa)
            ensure_financial_address(test_data["us_ca_id"], fa)
            currs = [(c or "").lower() for c in ((fa.get("storage") or {}).get("holds_currencies") or [])]
            if "usd" in currs and fa.get("status") != "closed":
                usd_ids.append(fa["id"])
        known = test_data.get("us_ca_fa_usd_id")
        if known not in usd_ids and usd_ids:
            test_data["us_ca_fa_usd_id"] = usd_ids[0]
        elif known in usd_ids:
            test_data["us_ca_fa_usd_id"] = known
    else:
        print("No FA on US CA — creating USD storage FA + US bank address.")
        response = requests.post(
            "https://api.stripe.com/v2/money_management/financial_accounts",
            headers=v2_headers(stripe_account=test_data["us_ca_id"]),
            json={
                "type": "storage",
                "storage": {"holds_currencies": ["usd"]},
                "display_name": "US Hotel Business Account",
            },
        )
        fa = print_result(response, show_full=False)
        if fa:
            test_data["us_ca_fa_usd_id"] = fa["id"]
            print_financial_account(fa)
            ensure_financial_address(test_data["us_ca_id"], fa)

    print(f"\n📝 us_ca_fa_usd_id: {test_data.get('us_ca_fa_usd_id')}")


  2.1 List or create US CA Financial Account (USD) + address

ℹ️ Found 3 FA(s) on US CA — not creating another.

💰 FA fa_65VPxjXQJr5pJ4eBbtp16VP1eFQgSQW4NIGeZiDGVy46Jk
  display_name: US Hotel Business Account
  status:       open
  country:      US
  currencies:   ['usd']
  USD available: 0.00
  USD inbound_pending: 0.00
  USD outbound_pending: 0.00

🏦 Bank details for inbound transfer
  Financial Address ID: finaddr_61VPyaNaXwD9XEioM16VP1eFQgSQW4NIGeZiDGVy4Qhk
  Status:               active
  Currency:             USD
  Type:                 us_bank_account
  Bank name:            FIFTH THIRD BANK US
  Routing number:       071919133
  Account number:       27001007516421425
  Account holder:       Weiheng Wang

💰 FA fa_65VPxhEDhoiIiN2K55U16VP1eFQgSQW4NIGeZiDGVy4S6i
  display_name: US Hotel Business Account
  status:       open
  country:      US
  currencies:   ['usd']
  USD available: 0.00
  USD inbound_pending: 0.00
  USD outbound_pending: 0.00

🏦 Bank details for inbound transfe

In [5]:
print_section("2.2 - 2.3 Verify FA is under SPC / FDIC passthrough attributes")

if not test_data['us_ca_fa_usd_id']:
    print("⚠️ No FA ID. Run 2.1 first.")
else:
    response = requests.get(
        f"https://api.stripe.com/v2/money_management/financial_accounts/{test_data['us_ca_fa_usd_id']}",
        headers=v2_headers(stripe_account=test_data['us_ca_id'])
    )
    fa = print_result(response, show_full=True)
    print("\n🔎 Confirm currency = usd and storage attributes look standard for a US FA.")
    manual_check("2.2 Verify via go/views that the FA is associated with SPC/US banking infrastructure.")


  2.2 - 2.3 Verify FA is under SPC / FDIC passthrough attributes

Status: 200
✅ SUCCESS
{
  "id": "fa_65VPcT1j296iiAuzC1d16VP1eFQgSQW4NIGeZiDGVy4Emm",
  "object": "v2.money_management.financial_account",
  "balance": {
    "available": {
      "usd": {
        "value": 1000,
        "currency": "usd"
      }
    },
    "inbound_pending": {
      "usd": {
        "value": 0,
        "currency": "usd"
      }
    },
    "outbound_pending": {
      "usd": {
        "value": 0,
        "currency": "usd"
      }
    }
  },
  "country": "US",
  "created": "2026-09-16T08:48:07.749Z",
  "display_name": "US Hotel Business Account",
  "metadata": null,
  "other": null,
  "payments": null,
  "status": "open",
  "status_details": null,
  "storage": {
    "holds_currencies": [
      "usd"
    ]
  },
  "type": "storage",
  "livemode": true
}

🔎 Confirm currency = usd and storage attributes look standard for a US FA.
🔎 MANUAL CHECK REQUIRED (not exposed via public API):
2.2 Verify via go/views that 

---
# Section 3: Money Movement — External (US Rails)

Inbound ACH/wire into the CA's FA, and outbound ACH/wire/OBT out to an external US bank account.

In [6]:
print_section("Create US Financial Address (needed for 3.1/3.2 inbound)")

if not test_data.get("us_ca_fa_usd_id"):
    print("⚠️ No FA ID. Run Section 2 first.")
else:
    print("Listing the US CA FA address (create only if none exists).")
    fa = {"id": test_data["us_ca_fa_usd_id"], "country": "US", "storage": {"holds_currencies": ["usd"]}}
    ensure_financial_address(test_data["us_ca_id"], fa)


  Create US Financial Address (needed for 3.1/3.2 inbound)

Listing the US CA FA address (create only if none exists).

🏦 Bank details for inbound transfer
  Financial Address ID: finaddr_61VPcUAkH6xO0udfB16VP1eFQgSQW4NIGeZiDGVy49o8
  Status:               active
  Currency:             USD
  Type:                 us_bank_account
  Bank name:            FIFTH THIRD BANK US
  Routing number:       071919133
  Account number:       27001007507251922
  Account holder:       Weiheng Wang


### 3.1 / 3.2 Inbound ACH & Wire (manual in LIVE mode)
Send a real ACH transfer and a real wire to the routing/account number shown above. There is no test-helper credit call available in live mode.

For Stripe internal testing, use https://admin.corp.stripe.com/excelsior/CreateV2AdjustmentForEngTesting/excl_VGz5lp9eLuaFub/ to adjust balance. Approved by @balance-abstractions-run in #ir-wary-champion slack channel.


In [8]:
print_section("3.1 / 3.2 Inbound ACH + Wire (manual)")
print("⚠️ Initiate a real ACH transfer AND a real wire transfer to the account/routing number above.")
print("Recommended amount: $1-$5 each, reference 'LIVE_TEST'.")
print("Expected: standard US ACH timing (T+1/T+2) for ACH; same/next-day for wire.")
print("\nAfter sending, poll the balance below until funds land.")


  3.1 / 3.2 Inbound ACH + Wire (manual)

⚠️ Initiate a real ACH transfer AND a real wire transfer to the account/routing number above.
Recommended amount: $1-$5 each, reference 'LIVE_TEST'.
Expected: standard US ACH timing (T+1/T+2) for ACH; same/next-day for wire.

After sending, poll the balance below until funds land.


In [9]:
print_section("Check US CA FA balance")

if test_data['us_ca_fa_usd_id']:
    response = requests.get(
        f"https://api.stripe.com/v2/money_management/financial_accounts/{test_data['us_ca_fa_usd_id']}",
        headers=v2_headers(stripe_account=test_data['us_ca_id'])
    )
    fa = print_result(response, show_full=False)
    if fa and 'balance' in fa:
        for balance_type, currencies in fa['balance'].items():
            for currency, amt in currencies.items():
                print(f"  {currency.upper()} {balance_type}: ${amt['value']/100:.2f}")


  Check US CA FA balance

Status: 200
✅ SUCCESS
Id: fa_65VPcT1j296iiAuzC1d16VP1eFQgSQW4NIGeZiDGVy4Emm
Object: v2.money_management.financial_account
Status: open
Balance: {'available': {'usd': {'value': 1000, 'currency': 'usd'}}, 'inbound_pending': {'usd': {'value': 0, 'currency': 'usd'}}, 'outbound_pending': {'usd': {'value': 0, 'currency': 'usd'}}}
  USD available: $10.00
  USD inbound_pending: $0.00
  USD outbound_pending: $0.00


In [53]:
print_section("Create external US bank account (for 3.3 / 3.4 / 3.5 outbound)")

if not test_data['us_ca_id']:
    print("⚠️ No US CA ID.")
else:
    payload = {
        "account_number": "8312738332",     # REPLACE with a real US test bank account
        "routing_number": "026073150",         # REPLACE with a real routing number
        "bank_account_type": "checking",
        "currency": "usd",
    }

    response = requests.post(
        "https://api.stripe.com/v2/core/vault/us_bank_accounts",
        headers=v2_headers(stripe_account=test_data['us_ca_id']),
        json=payload
    )

    bank_account = print_result(response, show_full=False)

    if bank_account:
        test_data['us_ca_external_bank_id'] = bank_account['id']
        print(f"\n🏦 External US Bank Account ID: {test_data['us_ca_external_bank_id']}")


  Create external US bank account (for 3.3 / 3.4 / 3.5 outbound)

Status: 200
✅ SUCCESS
Id: usba_61VRnzi4Ub6Wv7vpo16VP1eFQgSQW4NIGeZiDGVy4DQW
Object: v2.core.vault.us_bank_account

🏦 External US Bank Account ID: usba_61VRnzi4Ub6Wv7vpo16VP1eFQgSQW4NIGeZiDGVy4DQW


### 3.3 Outbound Transfer (OBT) using US rails
This is the same `outbound_transfers` the cell exercises the OBT object over local and wire.

In [54]:
print_section("3.3 Outbound Wire from US CA FA -> external US bank")

if not test_data['us_ca_fa_usd_id'] or not test_data['us_ca_external_bank_id']:
    print("⚠️ Missing FA or external bank account. Complete prior cells first.")
else:
    payload = {
        "amount": {"value": 100, "currency": "usd"},
        "from": {"currency": "usd", "financial_account": test_data['us_ca_fa_usd_id']},
        "to": {"currency": "usd", "payout_method": test_data['us_ca_external_bank_id']},
        # "delivery_options": {"bank_account": "wire"},
        "description": "Live test outbound wire"
    }
    response = requests.post(
        "https://api.stripe.com/v2/money_management/outbound_transfers",
        headers=v2_headers(stripe_account=test_data['us_ca_id']),
        json=payload
    )
    print_result(response, show_full=False)


  3.3 Outbound Wire from US CA FA -> external US bank

Status: 200
✅ SUCCESS
Id: obt_65VRo06AMrDf7ZqpSMP16VP1eFQgSQW4NIGeZiDGVy4Ghc
Object: v2.money_management.outbound_transfer
Status: processing


---
# Section 4: Money Movement — Internal (FA↔FA Cross-Entity)

Core value prop of Treasury for Platforms (T4P): transfers between the NL platform's FA (STEL) and the US CA's FA (SPC), in both directions and both currency combos. Expect **T+1** availability and FX where currencies differ.

In [13]:
print_section("Create Platform FA - if not already created")

if test_data.get("platform_fa_id"):
    print(f"Using platform EUR FA from Section 2: {test_data['platform_fa_id']}")
elif not test_data.get("platform_account_id"):
    print("⚠️ No platform account ID.")
else:
    payload = {
        "type": "storage",
        "storage": {"holds_currencies": ["eur"]},
        "display_name": "Mews EU Platform FA (EUR)"
    }
    response = requests.post(
        "https://api.stripe.com/v2/money_management/financial_accounts",
        headers=v2_headers(stripe_account=test_data['platform_account_id']),
        json=payload
    )
    fa = print_result(response, show_full=False)
    if fa:
        test_data['platform_fa_id'] = fa['id']
        print(f"\n💰 Platform FA (EUR) ID: {test_data['platform_fa_id']}")
        ensure_financial_address(test_data['platform_account_id'], fa)


  Create Platform FA - if not already created

Using platform EUR FA from Section 2: fa_65TwcHvlu49mspR6PrC16TZ4MPQ79Clh3eLzGNN0GbQ6O8


In [52]:
print_section("4.1 Platform FA (EUR, STEL) -> CA FA (USD, SPC)")

if not test_data['platform_fa_id'] or not test_data['us_ca_fa_usd_id']:
    print("⚠️ Missing platform EUR FA or CA USD FA.")
else:
    # Cross-currency OBP: amount is the EUR debit. Quote first, then pay with the same from/to/amount.
    movement = {
        "amount": {"value": 100, "currency": "eur"},  # €1.00 debited from EUR balance
        "from": {"currency": "eur", "financial_account": test_data['platform_fa_id']},
        "to": {
            "currency": "usd",
            "payout_method": test_data['us_ca_fa_usd_id'],
            "recipient": test_data['us_ca_id'],
        },
    }

    print("Creating outbound payment quote (EUR debit → USD credit)...")
    quote_response = requests.post(
        "https://api.stripe.com/v2/money_management/outbound_payment_quotes",
        headers=v2_headers(),
        json=movement,
    )
    quote = print_result(quote_response, show_full=True)
    if quote:
        fx = quote.get("fx_quote") or {}
        credited = (quote.get("to") or {}).get("credited") or {}
        print(f"\n💱 Quote {quote.get('id')}")
        print(f"  EUR debit:  {(quote.get('from') or {}).get('debited')}")
        print(f"  USD credit: {credited}")
        print(f"  FX lock expires: {fx.get('lock_expires_at')}")
        print(f"  Rates: {fx.get('rates')}")

        payload = {
            **movement,
            "outbound_payment_quote": quote["id"],
            "description": "4.1 Cross-entity FA->FA EUR->USD (STEL->SPC)",
        }
        response = requests.post(
            "https://api.stripe.com/v2/money_management/outbound_payments",
            headers=v2_headers(),
            json=payload,
        )
        payment = print_result(response, show_full=True)
        if payment:
            print(f"\nfrom.debited: {(payment.get('from') or {}).get('debited')}")
            print(f"to.credited:  {(payment.get('to') or {}).get('credited')}")


  4.1 Platform FA (EUR, STEL) -> CA FA (USD, SPC)

Creating outbound payment quote (EUR debit → USD credit)...
Status: 400
❌ ERROR
{
  "error": {
    "code": "invalid_fields",
    "message": "Some fields in the request were invalid: 'to.payout_method: The ID is invalid. Please check the id before retrying the request.'",
    "request_log_url": "https://dashboard.stripe.com/logs/req_v2RyqejDSRpgY8G3C",
    "invalid_fields": [
      {
        "field": "to.payout_method",
        "message": "The ID is invalid. Please check the id before retrying the request."
      }
    ]
  }
}


In [17]:
print_section("4.2 CA FA (USD, SPC) -> Platform FA (EUR, STEL)")

if not test_data['platform_fa_eur_id'] or not test_data['us_ca_fa_usd_id']:
    print("⚠️ Missing platform EUR FA or CA USD FA.")
else:
    movement = {
        "amount": {"value": 100, "currency": "usd"},  # $1.00 debited from USD balance
        "from": {"currency": "usd", "financial_account": test_data['us_ca_fa_usd_id']},
        "to": {
            "currency": "eur",
            "payout_method": test_data['platform_fa_eur_id'],
            "recipient": test_data['platform_account_id'],
        },
    }

    print("Creating outbound payment quote (USD debit → EUR credit)...")
    quote_response = requests.post(
        "https://api.stripe.com/v2/money_management/outbound_payment_quotes",
        headers=v2_headers(stripe_account=test_data['us_ca_id']),
        json=movement,
    )
    quote = print_result(quote_response, show_full=True)
    if quote:
        payload = {
            **movement,
            "outbound_payment_quote": quote["id"],
            "description": "4.2 Reverse cross-entity FA->FA USD->EUR (SPC->STEL)",
        }
        response = requests.post(
            "https://api.stripe.com/v2/money_management/outbound_payments",
            headers=v2_headers(stripe_account=test_data['us_ca_id']),
            json=payload,
        )
        print_result(response, show_full=True)
    print("\n🔎 If this errors, document it — reverse-direction (CA-initiated) cross-entity transfers may not be supported yet; that's exactly what this test is checking.")


  4.2 CA FA (USD, SPC) -> Platform FA (EUR, STEL)

Creating outbound payment quote (USD debit → EUR credit)...
Status: 400
❌ ERROR
{
  "error": {
    "code": "invalid_fields",
    "message": "Some fields in the request were invalid: 'to.payout_method: The ID is invalid. Please check the id before retrying the request.'",
    "request_log_url": "https://dashboard.stripe.com/logs/req_v227Lboekbwakr39N",
    "invalid_fields": [
      {
        "field": "to.payout_method",
        "message": "The ID is invalid. Please check the id before retrying the request."
      }
    ]
  }
}

🔎 If this errors, document it — reverse-direction (CA-initiated) cross-entity transfers may not be supported yet; that's exactly what this test is checking.


In [19]:
print_section("4.3 Platform FA (USD, STEL) -> CA FA (USD, SPC) - same currency")

if not test_data['platform_fa_usd_id']:
    print("⚠️ Platform FA (USD) not set. Create it first if the platform is multicurrency-enabled:")
    print("POST /v2/money_management/financial_accounts with storage.holds_currencies=['usd'], stripe_account=platform_account_id")
elif not test_data['us_ca_fa_usd_id']:
    print("⚠️ No CA USD FA.")
else:
    payload = {
        "amount": {"value": 100, "currency": "usd"},
        "from": {"currency": "usd", "financial_account": test_data['platform_fa_usd_id']},
        "to": {
            "currency": "usd",
            "payout_method": test_data['us_ca_fa_usd_id'],
            "recipient": test_data['us_ca_id']
        },
        "description": "4.3 Same-currency cross-entity FA->FA USD->USD (STEL->SPC)"
    }
    response = requests.post(
        "https://api.stripe.com/v2/money_management/outbound_payments",
        headers=v2_headers(stripe_account=test_data['platform_account_id']),
        json=payload
    )
    print_result(response, show_full=True)


  4.3 Platform FA (USD, STEL) -> CA FA (USD, SPC) - same currency

Status: 200
✅ SUCCESS
{
  "id": "obp_65VQM93vaY46nrYzHt016TZ4MPQ79Clh3eLzGNN0GbQLFQ",
  "object": "v2.money_management.outbound_payment",
  "amount": {
    "value": 100,
    "currency": "usd"
  },
  "cancelable": false,
  "created": "2026-09-18T09:34:35.115Z",
  "delivery_options": null,
  "description": "4.3 Same-currency cross-entity FA->FA USD->USD (STEL->SPC)",
  "expected_arrival_date": "2026-09-18T09:34:35.115Z",
  "from": {
    "debited": {
      "value": 100,
      "currency": "usd"
    },
    "financial_account": "fa_65TwcHvlu49mspR6PrC16TZ4MPQ79Clh3eLzGNN0GbQ6O8"
  },
  "metadata": {},
  "outbound_payment_quote": null,
  "payout_intent": null,
  "purpose": null,
  "receipt_url": "https://payments.stripe.com/transaction_receipt/session_61VQM93yB6mnUviNj16TZ4MPQ79Clh3eLzGNN0GbQMLI",
  "recipient_notification": {
    "setting": "configured"
  },
  "recipient_verification": null,
  "statement_descriptor": "BISCOF

In [20]:
print_section("4.4 - 4.5 Verify ledger entity tagging + T+1 timing")

if test_data['us_ca_fa_usd_id']:
    response = requests.get(
        "https://api.stripe.com/v2/money_management/transactions",
        headers=v2_headers(stripe_account=test_data['us_ca_id']),
        params={"financial_account": test_data['us_ca_fa_usd_id']}
    )
    txns = print_result(response, show_full=True)
    print("\n🔎 For each cross-entity transaction: check the entity tag on the balance transaction (should distinguish STEL vs SPC),")
    print("and compare 'created' vs 'posted'/'available_on' timestamps — expect T+1, not instant.")

manual_check("4.4 Intercompany ledger entry correctness is confirmed via intercompany sweep records, not this API.")


  4.4 - 4.5 Verify ledger entity tagging + T+1 timing

Status: 200
✅ SUCCESS
{
  "data": [
    {
      "id": "trxn_65VQM96tI7OQMOdbGAB16VP1eFQgSQW4NIGeZiDGVy4Lqa",
      "object": "v2.money_management.transaction",
      "amount": {
        "value": 100,
        "currency": "usd"
      },
      "balance_impact": {
        "available": {
          "value": 100,
          "currency": "usd"
        },
        "inbound_pending": {
          "value": 0,
          "currency": "usd"
        },
        "outbound_pending": {
          "value": 0,
          "currency": "usd"
        }
      },
      "category": "received_credit",
      "counterparty": null,
      "created": "2026-09-18T09:34:37.276Z",
      "description": "BISCOFF.IO",
      "financial_account": "fa_65VPcT1j296iiAuzC1d16VP1eFQgSQW4NIGeZiDGVy4Emm",
      "flow": {
        "received_credit": "rc_61VQM953A6sgHpVQ416VP1eFQgSQW4NIGeZiDGVy4E36",
        "type": "received_credit"
      },
      "status": "posted",
      "status_transi

---
# Section 5: Payments — Destination Charges + OBO

Mews's existing v1 Payments Balance / destination-charge model is unchanged by this pilot — confirm it still works OBO the US CA, and that funds can move from the v1 Payment Balance into the v2 FA.

In [27]:
print_section("5.1 Create destination charge OBO US CA")

if not test_data['us_ca_id']:
    print("⚠️ No US CA ID.")
elif not test_data['test_payment_method_id'] or not test_data['test_customer_id']:
    print("⚠️ LIVE mode requires a real payment method + customer.")
    print("Set test_data['test_customer_id'] and test_data['test_payment_method_id'] (a real card via Stripe.js / a real test purchase) before running this cell.")
else:
    response = requests.post(
        "https://api.stripe.com/v1/payment_intents",
        auth=v1_auth(),
        data={
            "amount": 50,  # $0.50
            "currency": "usd",
            "customer_account": test_data['test_customer_id'],
            "payment_method": test_data['test_payment_method_id'],
            "confirm": "true",
            "off_session": "true",
            "on_behalf_of": test_data['us_ca_id'],
            "transfer_data[destination]": test_data['us_ca_id'],
        }
    )
    pi = print_result(response, show_full=False)
    if pi:
        test_data['destination_charge_id'] = pi['id']
        print(f"\n💳 PaymentIntent ID: {test_data['destination_charge_id']}")


  5.1 Create destination charge OBO US CA

Status: 200
✅ SUCCESS
Id: pi_3UGyQpEW60zLc0T71SIvdXrK
Object: payment_intent
Status: succeeded

💳 PaymentIntent ID: pi_3UGyQpEW60zLc0T71SIvdXrK


In [28]:
print_section("5.2 / 5.4 Verify processing entity + statement descriptor")

if not test_data['destination_charge_id']:
    print("⚠️ No charge ID. Run 5.1 first.")
else:
    response = requests.get(
        f"https://api.stripe.com/v1/payment_intents/{test_data['destination_charge_id']}",
        auth=v1_auth()
    )
    pi = print_result(response, show_full=True)
    if pi:
        charges = pi.get('charges', {}).get('data', [])
        if charges:
            print(f"\n5.4 Statement descriptor: {charges[0].get('statement_descriptor', 'N/A')}")

manual_check("5.2 Confirm the balance transaction's processing entity = SPC via go/views.")


  5.2 / 5.4 Verify processing entity + statement descriptor

Status: 200
✅ SUCCESS
{
  "id": "pi_3UGyQpEW60zLc0T71SIvdXrK",
  "object": "payment_intent",
  "allowed_payment_method_types": null,
  "amount": 50,
  "amount_capturable": 0,
  "amount_details": {
    "tip": {}
  },
  "amount_received": 50,
  "application": null,
  "application_fee_amount": null,
  "automatic_payment_methods": {
    "allow_redirects": "always",
    "enabled": true
  },
  "canceled_at": null,
  "cancellation_reason": null,
  "capture_method": "automatic_async",
  "client_secret": "pi_3UGyQpEW60zLc0T71SIvdXrK_secret_B4P4sgmGl78hJRQRlCQqRb1J4",
  "confirmation_method": "automatic",
  "created": 1789725011,
  "currency": "usd",
  "customer": "cus_VHXNsK3FyeVVzc",
  "customer_account": "acct_1UGyIcEaCOBZn9dt",
  "description": null,
  "excluded_payment_method_types": null,
  "last_payment_error": null,
  "latest_charge": "ch_3UGyQpEW60zLc0T71XKh0h0V",
  "livemode": true,
  "managed_payments": {
    "enabled": fal

In [31]:
print_section("5.3 Transfer from v1 Payments Balance -> v2 FA (US CA)")

# Payments Balance → FA is a v1 Payout with payout_method = the v2 FA id.
# Do not use outbound_transfers: that API moves money *out of* an FA (from + to.payout_method).
# Docs: POST /v1/payouts + Stripe-Account = CA, payout_method = fa_...
ca_id = test_data.get("us_ca_id")
fa_id = test_data.get("us_ca_fa_usd_id")
amount = 50  # $0.50 in cents

if not ca_id or not fa_id:
    print("⚠️ No US CA ID or FA. Run Sections 1–2 first.")
else:
    v1_headers = {
        "Stripe-Account": ca_id,
        "Stripe-Version": API_VERSION,
    }

    print("Checking US CA payments balance...")
    bal_response = requests.get(
        "https://api.stripe.com/v1/balance",
        auth=v1_auth(),
        headers=v1_headers,
    )
    balance = print_result(bal_response, show_full=True)
    available_usd = 0
    pending_usd = 0
    if balance:
        for bucket in balance.get("available") or []:
            if bucket.get("currency") == "usd":
                available_usd = bucket.get("amount") or 0
        for bucket in balance.get("pending") or []:
            if bucket.get("currency") == "usd":
                pending_usd = bucket.get("amount") or 0
        print(f"\nUSD available: {available_usd / 100:.2f}  pending: {pending_usd / 100:.2f}")

    if available_usd < amount:
        print(f"⚠️ Available USD ({available_usd / 100:.2f}) is less than payout {amount / 100:.2f}.")
        print("   Run 5.1 (destination charge) first and wait until funds are available, then retry.")
    else:
        print(f"\nCreating v1 payout of {amount / 100:.2f} USD → FA {fa_id}...")
        response = requests.post(
            "https://api.stripe.com/v1/payouts",
            auth=v1_auth(),
            headers=v1_headers,
            data={
                "amount": amount,
                "currency": "usd",
                "payout_method": fa_id,
                "description": "5.3 Move funds from v1 Payments Balance into v2 FA",
            },
        )
        payout = print_result(response, show_full=True)
        if payout:
            print("\nFA balance after payout:")
            fa_response = requests.get(
                f"https://api.stripe.com/v2/money_management/financial_accounts/{fa_id}",
                headers=v2_headers(stripe_account=ca_id),
            )
            fa = print_result(fa_response, show_full=False)
            if fa:
                print_financial_account(fa)


  5.3 Transfer from v1 Payments Balance -> v2 FA (US CA)

Checking US CA payments balance...
Status: 200
✅ SUCCESS
{
  "object": "balance",
  "available": [
    {
      "amount": 0,
      "currency": "usd",
      "source_types": {
        "card": 0
      }
    }
  ],
  "instant_available": [
    {
      "amount": 50,
      "currency": "usd",
      "source_types": {
        "card": 50
      }
    }
  ],
  "livemode": true,
  "pending": [
    {
      "amount": 50,
      "currency": "usd",
      "source_types": {
        "card": 50
      }
    }
  ],
  "refund_and_dispute_prefunding": {
    "available": [
      {
        "amount": 0,
        "currency": "usd"
      }
    ],
    "pending": [
      {
        "amount": 0,
        "currency": "usd"
      }
    ]
  },
  "risk_reserved": {
    "available": [
      {
        "amount": 0,
        "currency": "usd"
      }
    ],
    "pending": [
      {
        "amount": 0,
        "currency": "usd"
      }
    ]
  }
}

USD available: 0.00  pend

---
# Section 6: Cross-Border Transfers (CBT)

Platform (NL) initiates a Connect transfer to the US CA using CBT mechanics, both linked (via `source_transaction`) and unlinked.

In [33]:
print_section("6.1 / 6.3 Platform creates linked transfer to US CA (source_transaction)")

if not test_data['destination_charge_id']:
    print("⚠️ No source charge available. Run Section 5 first, or supply a charge ID to link against.")
else:
    # Look up the underlying charge ID for the PaymentIntent to use as source_transaction
    response = requests.get(
        f"https://api.stripe.com/v1/payment_intents/{test_data['destination_charge_id']}",
        auth=v1_auth()
    )
    pi = print_result(response, show_full=False)
    source_charge_id = None
    if pi:
        charges = pi.get('charges', {}).get('data', [])
        source_charge_id = charges[0]['id'] if charges else None

    if not source_charge_id:
        print("⚠️ Could not resolve underlying charge id for linked transfer.")
    else:
        response = requests.post(
            "https://api.stripe.com/v1/transfers",
            auth=v1_auth(),
            data={
                "amount": 500,
                "currency": "usd",
                "destination": test_data['us_ca_id'],
                "source_transaction": source_charge_id,
                "description": "6.1/6.3 CBT linked transfer NL platform -> US CA"
            }
        )
        transfer = print_result(response, show_full=False)
        if transfer:
            test_data['cbt_transfer_linked_id'] = transfer['id']
            print(f"\n✅ Linked CBT Transfer ID: {test_data['cbt_transfer_linked_id']}")


  6.1 / 6.3 Platform creates linked transfer to US CA (source_transaction)

Status: 200
✅ SUCCESS
Id: pi_3UGyQpEW60zLc0T71SIvdXrK
Object: payment_intent
Status: succeeded
⚠️ Could not resolve underlying charge id for linked transfer.


In [35]:
print_section("6.4 Platform creates unlinked transfer to US CA (no source_transaction)")

if not test_data['us_ca_id']:
    print("⚠️ No US CA ID.")
else:
    response = requests.post(
        "https://api.stripe.com/v1/transfers",
        auth=v1_auth(),
        data={
            "amount": 100,
            "currency": "usd",
            "destination": test_data['us_ca_id'],
            "description": "6.4 CBT unlinked transfer NL platform -> US CA"
        }
    )
    transfer = print_result(response, show_full=False)
    if transfer:
        test_data['cbt_transfer_unlinked_id'] = transfer['id']
        print(f"\n✅ Unlinked CBT Transfer ID: {test_data['cbt_transfer_unlinked_id']}")


  6.4 Platform creates unlinked transfer to US CA (no source_transaction)

Status: 400
❌ ERROR
{
  "error": {
    "code": "balance_insufficient",
    "doc_url": "https://stripe.com/docs/error-codes/balance-insufficient",
    "message": "You have insufficient funds in your Stripe account. One likely reason you have insufficient funds is that your funds are automatically being paid out; try enabling manual payouts by going to https://dashboard.stripe.com/account/payouts.",
    "request_log_url": "https://dashboard.stripe.com/acct_1SPgP3EW60zLc0T7/workbench/logs?object=req_GA91jxYx8tj1tD",
    "type": "invalid_request_error"
  }
}


In [36]:
print_section("6.2 Verify CBT fee applied (25bps for US corridor)")

transfer_id = test_data['cbt_transfer_linked_id'] or test_data['cbt_transfer_unlinked_id']
if not transfer_id:
    print("⚠️ No CBT transfer to check. Run 6.1/6.3 or 6.4 first.")
else:
    response = requests.get(
        f"https://api.stripe.com/v1/transfers/{transfer_id}",
        auth=v1_auth()
    )
    transfer = print_result(response, show_full=True)
    bt_id = transfer.get('balance_transaction') if transfer else None
    if bt_id:
        bt_response = requests.get(
            f"https://api.stripe.com/v1/balance_transactions/{bt_id}",
            auth=v1_auth()
        )
        bt = print_result(bt_response, show_full=True)
        if bt:
            print(f"\n💰 Fee: {bt.get('fee', 'N/A')} (expect ~25bps of transfer amount for the US corridor)")


  6.2 Verify CBT fee applied (25bps for US corridor)

⚠️ No CBT transfer to check. Run 6.1/6.3 or 6.4 first.


---
# Section 7: Recipient / Payouts

US CA adds an external US bank account as a payout method, then pays out from both the v1 Payments Balance and the v2 FA. No Confirmation of Payee (CoP/VoP) is expected in the US (that's UK-specific).

In [37]:
print_section("7.1 US CA adds US bank account as payout method")
print("Reuses the external bank account created for Section 3 (test_data['us_ca_external_bank_id']),")
print("or re-run the vault creation cell in Section 3 if you need a distinct payout method.")
print(f"\nCurrent value: {test_data['us_ca_external_bank_id']}")


  7.1 US CA adds US bank account as payout method

Reuses the external bank account created for Section 3 (test_data['us_ca_external_bank_id']),
or re-run the vault creation cell in Section 3 if you need a distinct payout method.

Current value: ba_1UGaEfIjc7BMQplWfckiCBCC


In [50]:
print_section("7.2 US CA payout from Payment Balance -> US bank (ACH)")

if not test_data['us_ca_id']:
    print("⚠️ No US CA ID.")
else:
    response = requests.post(
        "https://api.stripe.com/v1/payouts",
        auth=v1_auth(),
        headers={"Stripe-Account": test_data['us_ca_id']},
        data={
            "amount": 100,
            "currency": "usd",
            "method": "standard",
            "description": "7.2 Live test payout from Payments Balance"
        }
    )
    print_result(response, show_full=False)


  7.2 US CA payout from Payment Balance -> US bank (ACH)

Status: 400
❌ ERROR
{
  "error": {
    "code": "balance_insufficient",
    "doc_url": "https://stripe.com/docs/error-codes/balance-insufficient",
    "message": "You have insufficient funds in your Stripe account for this transfer. Your card balance is too low.  You can use the /v1/balance endpoint to view your Stripe balance (for more details, see stripe.com/docs/api#balance).",
    "request_log_url": "https://dashboard.stripe.com/acct_1SPgP3EW60zLc0T7/workbench/logs?object=req_OobbV7AftBBU4n",
    "type": "invalid_request_error"
  }
}


In [51]:
print_section("7.3 US CA payout from v2 FA -> US bank (OBT)")

if not test_data['us_ca_fa_usd_id'] or not test_data['us_ca_external_bank_id']:
    print("⚠️ Missing FA or external bank account.")
else:
    payload = {
        "amount": {"value": 50, "currency": "usd"},
        "from": {"currency": "usd", "financial_account": test_data['us_ca_fa_usd_id']},
        "to": {"currency": "usd", "payout_method": test_data['us_ca_external_bank_id']},
        "description": "7.3 OBT payout from v2 FA"
    }
    response = requests.post(
        "https://api.stripe.com/v2/money_management/outbound_transfers",
        headers=v2_headers(stripe_account=test_data['us_ca_id']),
        json=payload
    )
    print_result(response, show_full=False)

print("\n🔎 7.4 Confirm no Confirmation of Payee / Verification of Payee step was triggered anywhere above — that's UK-only.")


  7.3 US CA payout from v2 FA -> US bank (OBT)

Status: 400
❌ ERROR
{
  "error": {
    "code": "invalid_fields",
    "message": "Some fields in the request were invalid: 'to.payout_method: The ID is invalid. Please check the id before retrying the request.'",
    "request_log_url": "https://dashboard.stripe.com/logs/req_v2U8UuabqlDJHZRWb",
    "invalid_fields": [
      {
        "field": "to.payout_method",
        "message": "The ID is invalid. Please check the id before retrying the request."
      }
    ]
  }
}

🔎 7.4 Confirm no Confirmation of Payee / Verification of Payee step was triggered anywhere above — that's UK-only.


---
# Section 8: Application Fees (AFF) — NL Platform, US CAs

Proves Mews (the NL platform) can monetise US CA transactions via `application_fee_amount`, with fees landing in the platform's balance, not the CA's.

In [43]:
print_section("8.1 Destination charge with application_fee_amount on US CA")

if not test_data['test_payment_method_id'] or not test_data['test_customer_id']:
    print("⚠️ LIVE mode requires a real payment method + customer (see Section 5.1).")
else:
    response = requests.post(
        "https://api.stripe.com/v1/payment_intents",
        auth=v1_auth(),
        data={
            "amount": 50,  # $0.50
            "currency": "usd",
            "customer_account": test_data['test_customer_id'],
            "payment_method": test_data['test_payment_method_id'],
            "confirm": "true",
            "off_session": "true",
            "on_behalf_of": test_data['us_ca_id'],
            "transfer_data[destination]": test_data['us_ca_id'],
            "application_fee_amount": 100  # $1.00 platform fee
        }
    )
    pi = print_result(response, show_full=True)
    if pi:
        print(f"\n💰 Application fee amount requested: $1.00 on a $10.00 charge")


  8.1 Destination charge with application_fee_amount on US CA

Status: 400
❌ ERROR
{
  "error": {
    "message": "The provided PaymentMethod was previously used with a PaymentIntent without Customer attachment, shared with a connected account without Customer attachment, or was detached from a Customer. It may not be used again. To use a PaymentMethod multiple times, you must attach it to a Customer first.",
    "request_log_url": "https://dashboard.stripe.com/acct_1SPgP3EW60zLc0T7/workbench/logs?object=req_vLAawdnKnwHMNS",
    "type": "invalid_request_error"
  }
}


In [44]:
print_section("8.2 / 8.5 Verify application fee currency + lands in Platform's balance")

response = requests.get(
    "https://api.stripe.com/v1/application_fees",
    auth=v1_auth(),
    params={"limit": 5}
)
fees = print_result(response, show_full=True)

response = requests.get("https://api.stripe.com/v1/balance", auth=v1_auth())
balance = print_result(response, show_full=True)

manual_check("8.3 Verify the fee was billed to the correct entity (platform pays Connect fees) via User Billing ledger.")


  8.2 / 8.5 Verify application fee currency + lands in Platform's balance

Status: 200
✅ SUCCESS
{
  "object": "list",
  "count": 0,
  "data": [],
  "has_more": false,
  "url": "/v1/application_fees"
}
Status: 200
✅ SUCCESS
{
  "object": "balance",
  "available": [
    {
      "amount": 21,
      "currency": "cad",
      "source_types": {
        "card": 21
      }
    },
    {
      "amount": -46,
      "currency": "eur",
      "source_types": {
        "card": -46
      }
    },
    {
      "amount": -147,
      "currency": "aud",
      "source_types": {
        "card": -147
      }
    },
    {
      "amount": 0,
      "currency": "usd",
      "source_types": {
        "card": 0
      }
    },
    {
      "amount": 0,
      "currency": "gbp",
      "source_types": {
        "card": 0
      }
    }
  ],
  "connect_reserved": [
    {
      "amount": 0,
      "currency": "cad"
    },
    {
      "amount": 0,
      "currency": "eur"
    },
    {
      "amount": 0,
      "currency": "au

In [45]:
print_section("8.4 Separate Charge & Transfer (SCT) with application fee")

if not test_data['test_payment_method_id'] or not test_data['test_customer_id']:
    print("⚠️ LIVE mode requires a real payment method + customer.")
else:
    # Step 1: charge on the platform itself (no destination)
    charge_response = requests.post(
        "https://api.stripe.com/v1/payment_intents",
        auth=v1_auth(),
        data={
            "amount": 50,  # $0.50
            "currency": "usd",
            "customer": test_data['test_customer_id'],
            "payment_method": test_data['test_payment_method_id'],
            "confirm": "true",
            "off_session": "true"
        }
    )
    pi = print_result(charge_response, show_full=False)
    charge_id = None
    if pi:
        charges = pi.get('charges', {}).get('data', [])
        charge_id = charges[0]['id'] if charges else None
        test_data['sct_charge_id'] = pi['id']

    if not charge_id:
        print("⚠️ Could not resolve underlying charge id.")
    else:
        # Step 2: transfer $9.00 to the CA, keeping $1.00 as the platform's fee
        transfer_response = requests.post(
            "https://api.stripe.com/v1/transfers",
            auth=v1_auth(),
            data={
                "amount": 900,  # $9.00 (i.e. $10.00 charge - $1.00 fee retained)
                "currency": "usd",
                "destination": test_data['us_ca_id'],
                "source_transaction": charge_id,
                "description": "8.4 SCT with application fee retained by platform"
            }
        )
        transfer = print_result(transfer_response, show_full=False)
        if transfer:
            test_data['sct_transfer_id'] = transfer['id']
            print(f"\n✅ SCT Transfer ID: {test_data['sct_transfer_id']} ($9.00 to CA, $1.00 retained as fee)")


  8.4 Separate Charge & Transfer (SCT) with application fee

Status: 400
❌ ERROR
{
  "error": {
    "message": "The customer supplied is not valid. Accounts configured as a Customer should be provided in the customer_account field.",
    "param": "customer",
    "request_log_url": "https://dashboard.stripe.com/acct_1SPgP3EW60zLc0T7/workbench/logs?object=req_A7B1pROAHBXmve",
    "type": "invalid_request_error"
  }
}
⚠️ Could not resolve underlying charge id.


---
# Post-Test Validation

In [47]:
manual_check(
    "- User Billing: fees charged to correct entity -> check UB ledger for the platform\n"
    "- Intercompany: STEL↔SPC entries created correctly -> check intercompany sweeps / ledger\n"
    "- Safeguarding: no breach from cross-entity transfer -> confirm safeguarding KPI not violated\n"
    "- Compliance: US CA under correct CP throughout -> go/views re-check post all transactions"
)

🔎 MANUAL CHECK REQUIRED (not exposed via public API):
- User Billing: fees charged to correct entity -> check UB ledger for the platform
- Intercompany: STEL↔SPC entries created correctly -> check intercompany sweeps / ledger
- Safeguarding: no breach from cross-entity transfer -> confirm safeguarding KPI not violated
- Compliance: US CA under correct CP throughout -> go/views re-check post all transactions

Record the result in the ✅/❌ column of the test plan doc.


## Success Criteria
All scenarios ✅ = Production validated for the NL→US corridor (T4P).

**Key gates:**
- Section 1 (onboarding) must all pass — proves US CPs apply
- Section 4 (cross-entity FA↔FA) must pass — core T4P value prop
- Section 8 (AFF) must pass — proves Mews can monetise

In [48]:
print_section("Test Data Summary")

print("\n📝 IDs Created / Used During Testing:\n")
for key, value in test_data.items():
    print(f"{key:30s}: {value if value else '(not set)'}")

print("\n" + "="*80)
print("  NL -> US TREASURY FOR PLATFORMS LIVE TEST COMPLETE")
print("="*80)
print("\nNext steps:")
print("  - Fill in the ✅/❌ column of the source test plan doc for every scenario")
print("  - Compare results against the sandbox penny test (20/20 passing, 2026-07-08)")
print("  - Escalate any failures/divergence to #spend-and-earn (SPEAR)")
print("  - Confirm T+1 settlement timing for cross-entity transfers with Megan Li")


  Test Data Summary


📝 IDs Created / Used During Testing:

platform_account_id           : acct_1SPgP3EW60zLc0T7
platform_fa_id                : fa_65TwcHvlu49mspR6PrC16TZ4MPQ79Clh3eLzGNN0GbQ6O8
platform_fa_eur_id            : fa_65TwcHvlu49mspR6PrC16TZ4MPQ79Clh3eLzGNN0GbQ6O8
platform_fa_usd_id            : fa_65TwcHvlu49mspR6PrC16TZ4MPQ79Clh3eLzGNN0GbQ6O8
platform_fa_gbp_id            : fa_65TwcHvlu49mspR6PrC16TZ4MPQ79Clh3eLzGNN0GbQ6O8
us_ca_id                      : acct_1UFdgtIjc7BMQplW
us_ca_id1                     : acct_1Tbg1nINWqC43lYI
us_ca_fa_usd_id               : fa_65VPcT1j296iiAuzC1d16VP1eFQgSQW4NIGeZiDGVy4Emm
us_ca_external_bank_id        : ba_1UGaEfIjc7BMQplWfckiCBCC
test_payment_method_id        : pm_1UGyMWEW60zLc0T7kUPJ2UKx
test_customer_id              : acct_1UGyIcEaCOBZn9dt
destination_charge_id         : pi_3UGyQpEW60zLc0T71SIvdXrK
cbt_transfer_linked_id        : (not set)
cbt_transfer_unlinked_id      : (not set)
sct_charge_id                 : (not set)
sct_tra